In 20260127_pairwise_to_homeo_weights.ipynb, we found the best weight for each pairwise comparison in respect to homeostatic need. We found a range of weights that can be used for each objective term without penalizing or sacrificing the homeostatic need. Now we want to find the best combination of weights for all terms together. In this notebook, I will be using the epsilon constraint method to find the best combination of weights for all terms together. We have a vague idea of the order of importance of the terms: we value homeostatic objective the most, then kinetics, then efficiency, secretion, and diversity are kind of equal. Secondly, through pareto_exploration.py, we found several sets of weight combinations that can be used without sacrificing the homeostatic need, as an example: lam_sec = 4.07e-10, lam_eff = 8.56e-7, lam_kin = 1.60e-3, lam_div=2.57e-7. We will use these as a prior for our epsilon constraint method.

We will fix the homeostatic term to be within a certain range of the optimal value, and then we will vary the weights of the other terms to find the best combination of weights that maximizes the other terms while still satisfying the homeostatic constraint.

In [1]:
import altair
import pandas as pd
import os, dill
import numpy as np
import cvxpy as cp

from ecoli.processes.metabolism_redux_classic import NetworkFlowModel, FlowResult

os.chdir(os.path.expanduser('~/dev/vEcoli'))

%load_ext autoreload

In [2]:
def load_sim(
        time_num: int,
        date: str,
        experiment_name: str,
        condition: str,
        experiment_type: str,
):
    # --- Load Sim ---
    time = str(time_num)
    entry = f'{experiment_name}_{time}_{date}'
    folder_path = f'out/{experiment_type}/{condition}/{entry}/'

    output = np.load(folder_path + '0_output.npy',allow_pickle='TRUE').item()
    output = output['agents']['0']
    fba = output['listeners']['fba_results']
    bulk = pd.DataFrame(output['bulk'])
    f = open(folder_path + 'agent_steps.pkl', 'rb')
    agent = dill.load(f)
    f.close()

    metabolism = agent['ecoli-metabolism-redux-classic']

    return fba, bulk, metabolism, output

In [3]:
# import sim
time_num = 600
# date = '2026-01-22'
date = '2026-04-06'
experiment_name = 'homeostatic_only'
condition = 'basal'
experiment_type = 'objective_weight'

fba, bulk, metabolism, output = load_sim(time_num, date, experiment_name, condition, experiment_type)

In [4]:
stoichiometry = metabolism.stoichiometry.copy()
reaction_names = metabolism.reaction_names
metabolites = metabolism.metabolite_names.copy()
counts_to_molar = output['listeners']['enzyme_kinetics']['counts_to_molar']

homeostatic_only_term = np.array(fba['homeostatic_term'].copy())/counts_to_molar # covert to counts
homeostatic_only_baseline = np.max(homeostatic_only_term) # in counts

homeostatic_dm_targets =  pd.DataFrame(fba['target_homeostatic_dmdt'], columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0).max(axis=0)
homeostatic_metabolite_counts = pd.DataFrame(fba['homeostatic_metabolite_counts'], columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0).mean(axis=0)
maintenance = pd.DataFrame(fba["maintenance_target"][1:], columns=['maintenance_reaction']).mul(counts_to_molar[1:], axis=0).mean(axis=0)
kinetic = pd.DataFrame(fba["target_kinetic_fluxes"], columns=metabolism.kinetic_constraint_reactions).mul(counts_to_molar, axis=0).mean(axis=0)

In [5]:
homeostatic_dm_targets =  pd.DataFrame(fba['target_homeostatic_dmdt'], columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0).iloc[1]
homeostatic_metabolite_counts = pd.DataFrame(fba['homeostatic_metabolite_counts'], columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0).iloc[1]
maintenance = pd.DataFrame(fba["maintenance_target"][1:], columns=['maintenance_reaction']).mul(counts_to_molar[1:], axis=0).iloc[1]
kinetic = pd.DataFrame(fba["target_kinetic_fluxes"], columns=metabolism.kinetic_constraint_reactions).mul(counts_to_molar, axis=0).iloc[1]

os.chdir(os.path.expanduser('~/dev/vEcoli/notebooks/Heena notebooks/Metabolism_New Genes'))
from pareto_exploration import run

df = run(
    stoichiometry=stoichiometry,
    metabolites=metabolites,
    reaction_names=reaction_names,
    metabolism=metabolism,
    homeostatic_metabolite_counts=homeostatic_metabolite_counts,
    homeostatic_dm_targets=homeostatic_dm_targets,
    kinetic=kinetic,
    maintenance=maintenance,
    counts_to_molar=counts_to_molar,
    n_samples=2000,
    n_jobs=10,
)

Sampling 2000 weight combinations (log-uniform in feasible ranges)...
Solving 2000 problems (10 parallel job(s))...


  2%|▏         | 30/2000 [00:03<04:27,  7.38it/s]/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
  6%|▋         | 130/2000 [00:20<04:59,  6.24it/s]/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
  8%|▊         | 150/2000 [00:23<05:08,  5.99it/s]/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
  8%|▊         | 160/2000 [00:24<04:22,  7.00it/s]/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invali

  2000 successful solves.
  Saved raw results: notebooks/Heena notebooks/Metabolism_New Genes/pareto_results_init_selective_2000samples/pareto_results.csv
Generating plots...
  Saved: notebooks/Heena notebooks/Metabolism_New Genes/pareto_results_init_selective_2000samples/pairwise_analysis.html
  Saved: notebooks/Heena notebooks/Metabolism_New Genes/pareto_results_init_selective_2000samples/parallel_coordinates.html
  Saved: notebooks/Heena notebooks/Metabolism_New Genes/pareto_results_init_selective_2000samples/pareto_3d.html
Done.
